# Training curves — live viewer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neumbilly/NextSearch/blob/cursor/lfm2.5-2.6b-stage1-3be6/notebooks/02_training_curves.ipynb)

View **training curves live, inline, with no external tracker**. Every stage of
the LFM experiment logs scalars and per-`step` metrics to a local
`metrics.jsonl` via `nextsearch.experiment.RunLogger`; this notebook reads that
file and plots one curve per series (`train/loss`, `train/reward`,
`eval/accuracy`, …), refreshing in place as a run writes more.

Stage 1 does no training yet, so the cell below optionally logs a short
**synthetic** curve so you can see the live view working right now. Point
`RUN_DIR` at a real run (local or on Google Drive) and the same view shows the
real curves an SFT/OPD/RL stage produces — no code change.

## 1 · Install

In [ ]:
import os
if not os.path.isdir("NextSearch"):
    !git clone https://github.com/neumbilly/NextSearch.git
%cd NextSearch
!git fetch --all --quiet && git checkout cursor/lfm2.5-2.6b-stage1-3be6 --quiet
!pip install --quiet -e ".[experiment]"
print("installed nextsearch from", os.getcwd())

## 2 · Point at a run

`RUN_DIR` is any directory that has (or will have) a `metrics.jsonl`. Use a
Google Drive path to watch a run that is training elsewhere; the viewer just
tails the file.

In [ ]:
from pathlib import Path

# e.g. "/content/drive/MyDrive/nextsearch-lfm/runs/lfm2.5-2.6b/<run_id>"
RUN_DIR = Path("/content/nextsearch-lfm/runs/demo")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("watching:", RUN_DIR / "metrics.jsonl")

## 3 · (Optional) log a synthetic curve so you can see it work now

Remove this once a real training stage is writing to `RUN_DIR/metrics.jsonl`.
It just demonstrates the exact call SFT/OPD/RL will make.

In [ ]:
import math, random, time
from nextsearch.experiment import RunLogger

# Reuse RUN_DIR as the run directory itself.
log = RunLogger(RUN_DIR.parent, experiment=RUN_DIR.parent.name, run_id=RUN_DIR.name)
for step in range(0, 501, 25):
    log.log(step=step, phase="train",
            loss=round(2.4 * math.exp(-step / 250) + random.uniform(0, 0.05), 4),
            reward=round(min(0.9, 0.1 + step / 650), 4),
            lr=2e-5)
    if step % 100 == 0:
        log.log(step=step, phase="eval",
                accuracy=round(min(0.75, 0.2 + step / 900), 4))
log.close()
print("logged synthetic curves to", log.metrics_path)

## 4 · Static snapshot of the curves

In [ ]:
%matplotlib inline
from nextsearch.experiment.viewer import render_curves
import matplotlib.pyplot as plt

render_curves(RUN_DIR, title=f"LFM experiment · {RUN_DIR.name}")
plt.show()

## 5 · Live view (redraws in place)

Run this while a training/eval job appends to `metrics.jsonl`. It refreshes
every few seconds and stops on the Colab stop button, after `max_seconds`, or
when no new step arrives for `stop_when_idle_s`.

In [ ]:
from nextsearch.experiment.viewer import live_curves

live_curves(RUN_DIR, refresh_s=3, stop_when_idle_s=20,
            title=f"LFM experiment · {RUN_DIR.name}")

## How stages feed this

Any stage logs curves with the same API — this is the whole contract:

```python
from nextsearch.experiment import RunLogger
log = RunLogger(OUTPUT_ROOT, experiment="lfm2.5-2.6b")
for step in training_loop:
    log.log(step=step, phase="train", loss=loss, reward=reward, lr=lr)
    if step % eval_every == 0:
        log.log(step=step, phase="eval", accuracy=acc)   # eval curves too
```

Because it is just a local `metrics.jsonl`, the curves survive a Colab
disconnect and this notebook can watch a run training on a Modal GPU, a rented
box, or your laptop — wherever the file lives (mount the Drive folder it writes
to). Stage-1 rollout telemetry shows up the same way via the four-panel
dashboard in `01_lfm_serving_and_harness.ipynb`.